In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()
PROC_DIR = PROJECT_DIR / "Data" / "Processed"

print("PROJECT_DIR:", PROJECT_DIR)
print("PROC_DIR   :", PROC_DIR)

PROJECT_DIR: /Users/linda/RiceDatathon_2026_Finance
PROC_DIR   : /Users/linda/RiceDatathon_2026_Finance/Data/Processed


In [2]:
panel = pd.read_csv(PROC_DIR / "panel_with_amenity_scores.csv")
print("panel:", panel.shape)

# Choose one:
WINDOW_TAG = "post"   # recommended first
# WINDOW_TAG = "pre"

TARGET_MAP = {
    "pre":  "revpar_growth_2015_2020_pct",
    "post": "revpar_growth_2022_2025_pct",
}
TARGET_SOURCE = TARGET_MAP[WINDOW_TAG]
TARGET_COL = "target"

if "time_window_tag" in panel.columns:
    df = panel[panel["time_window_tag"] == WINDOW_TAG].copy()
else:
    df = panel.copy()  # fallback (not ideal)
    print("Warning: time_window_tag not found; using all rows.")

df[TARGET_COL] = pd.to_numeric(df[TARGET_SOURCE], errors="coerce")
df = df[df[TARGET_COL].notna()].copy()

print("Window:", WINDOW_TAG, "| rows:", df.shape[0], "| target:", TARGET_SOURCE)

panel: (38941, 123)
Window: post | rows: 19457 | target: revpar_growth_2022_2025_pct


In [3]:
BASE_NUM_FEATURES = [
    "daily_needs_score",
    "daily_needs_score_equal",
    "lifestyle_score",
    "family_support_score",
    "drivetime_min",
    "numunits",
    "areaperunit",
    "yearbuilt",
    "year_renov",
    "ownrent_avg_rent",
    "ownrent_avg_mortgage",
    "ownrent_spread_abs",
    "ownrent_spread_pct",
    "supply_growth_pct",
    "supply_new_units",
    "supply_baseline_units",
]

FEATURES = [c for c in BASE_NUM_FEATURES if c in df.columns]
if len(FEATURES) < 5:
    raise RuntimeError(f"Too few features found: {FEATURES}")

ID_CANDIDATES = ["ubid", "property_id"]
GROUP_COL = next((c for c in ID_CANDIDATES if c in df.columns), None)

if GROUP_COL is None:
    stable_candidates = ["latitude", "longitude", "lat", "lon", "yearbuilt", "numunits"]
    stable = [c for c in stable_candidates if c in df.columns]
    if len(stable) == 0:
        raise RuntimeError("No ID and no stable columns to construct group key.")
    df["_group_key"] = df[stable].astype(str).agg("_".join, axis=1)
    GROUP_COL = "_group_key"

print("n_features:", len(FEATURES))
print("Using group column:", GROUP_COL)

n_features: 16
Using group column: ubid


In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

X = df[FEATURES].copy()
y = df[TARGET_COL].copy()
groups = df[GROUP_COL].astype(str).values

gkf = GroupKFold(n_splits=5)

In [5]:
CANDIDATES = [
    # Balanced baseline
    dict(num_leaves=63,  min_child_samples=30, subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0),
    dict(num_leaves=63,  min_child_samples=50, subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0),

    # Slightly stronger
    dict(num_leaves=127, min_child_samples=30, subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0),
    dict(num_leaves=127, min_child_samples=50, subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0),

    # More regularized / stable
    dict(num_leaves=63,  min_child_samples=80, subsample=0.80, colsample_bytree=0.80, reg_lambda=2.0),
    dict(num_leaves=127, min_child_samples=80, subsample=0.80, colsample_bytree=0.80, reg_lambda=2.0),

    # Optional: if you want 2 more, uncomment
    # dict(num_leaves=255, min_child_samples=60, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.0),
    # dict(num_leaves=255, min_child_samples=100, subsample=0.80, colsample_bytree=0.80, reg_lambda=3.0),
]

print("Number of candidates:", len(CANDIDATES))

Number of candidates: 6


In [6]:
def eval_candidate(params: dict) -> dict:
    model = LGBMRegressor(
        n_estimators=3500,
        learning_rate=0.02,
        random_state=42,
        **params
    )
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])

    rmses = []
    for tr_idx, va_idx in gkf.split(X, y, groups=groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pipe.fit(X_tr, y_tr)
        pred_va = pipe.predict(X_va)
        rmses.append(rmse(y_va, pred_va))

    return {
        **params,
        "rmse_mean": float(np.mean(rmses)),
        "rmse_std": float(np.std(rmses)),
    }

results = []
for i, params in enumerate(CANDIDATES, start=1):
    r = eval_candidate(params)
    results.append(r)
    print(f"[{i}/{len(CANDIDATES)}] rmse_mean={r['rmse_mean']:.5f} rmse_std={r['rmse_std']:.5f} params={params}")

res = pd.DataFrame(results).sort_values(["rmse_mean","rmse_std"]).reset_index(drop=True)
res

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3429
[LightGBM] [Info] Number of data points in the train set: 15565, number of used features: 16
[LightGBM] [Info] Start training from score -0.051480


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000732 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3425
[LightGBM] [Info] Number of data points in the train set: 15565, number of used features: 16
[LightGBM] [Info] Start training from score -0.050974


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000487 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3430
[LightGBM] [Info] Number of data points in the train set: 15566, number of used features: 16
[LightGBM] [Info] Start training from score -0.052151


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000678 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3427
[LightGBM] [Info] Number of data points in the train set: 15566, number of used features: 16
[LightGBM] [Info] Start training from score -0.052490


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[6/6] rmse_mean=0.12625 rmse_std=0.00511 params={'num_leaves': 127, 'min_child_samples': 80, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 2.0}


,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda,rmse_mean,rmse_std
0,63,30,0.85,0.85,1.0,0.124167,0.004680
1,63,50,0.85,0.85,1.0,0.124797,0.004854
2,63,80,0.80,0.80,2.0,0.125023,0.004728
3,127,30,0.85,0.85,1.0,0.125263,0.004873
4,127,50,0.85,0.85,1.0,0.126128,0.005192
5,127,80,0.80,0.80,2.0,0.126247,0.005113


In [10]:
best = res.iloc[0].to_dict()
print("Best candidate:")
print(best)

# ---- Enforce correct dtypes for LightGBM ----
INT_PARAMS = {"num_leaves", "min_child_samples", "max_depth", "n_estimators"}
FLOAT_PARAMS = {"subsample", "colsample_bytree", "reg_lambda", "reg_alpha", "min_split_gain"}

BEST_PARAMS = {}
for k in ["num_leaves","min_child_samples","subsample","colsample_bytree","reg_lambda"]:
    v = best[k]
    if k in INT_PARAMS:
        BEST_PARAMS[k] = int(round(float(v)))
    elif k in FLOAT_PARAMS:
        BEST_PARAMS[k] = float(v)
    else:
        BEST_PARAMS[k] = v

print("BEST_PARAMS (typed):", BEST_PARAMS)

Best candidate:
{'num_leaves': 63.0, 'min_child_samples': 30.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'rmse_mean': 0.12416736133762699, 'rmse_std': 0.004679806727887288}
BEST_PARAMS (typed): {'num_leaves': 63, 'min_child_samples': 30, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 1.0}


In [11]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

final_model = LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.015,
    random_state=42,
    **BEST_PARAMS
)

final_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", final_model),
])

final_pipe.fit(X, y)
print("Final model fitted for window:", WINDOW_TAG)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000761 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3434
[LightGBM] [Info] Number of data points in the train set: 19457, number of used features: 16
[LightGBM] [Info] Start training from score -0.051733
Final model fitted for window: post
